In [4]:
# Enhanced DataGen.py with Unified Incremental Output File (with change_type)
import pandas as pd
import numpy as np
from faker import Faker
import random
import os
from datetime import datetime, timedelta

# Initialize
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# Config
MODE = "full"  # Options: "full", "incremental"
OUTPUT_DIR = "./Source_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Table sizes
NUM_CUSTOMERS = 1000
NUM_PRODUCTS = 1000
NUM_ORDERS = 100000
NUM_ORDER_ITEMS = 200000
NUM_SUPPLIERS = 100
NUM_CATEGORIES = 50
NUM_PAYMENTS = 100000
NUM_REVIEWS = 20000
NUM_SHIPMENTS = 80000
NUM_RETURNS = 10000
NUM_INVENTORY = 1000
NUM_BROWSING = 40000
NUM_SUPPORT_TICKETS = 6000
NUM_DISCOUNTS = 100
NUM_REGIONS = 200

# Helpers
def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

def maybe_null(value, chance=0.05):
    return None if random.random() < chance else value

def incremental_template(name, base_df, gen_func, id_col):
    now = datetime.now()
    all_changes = []

    # Inserts
    start_id = base_df[id_col].max() + 1 if not base_df.empty else 1
    for i in range(start_id, start_id + 10):
        row = gen_func(i)
        row['change_type'] = 'I'
        row['last_modified'] = now
        all_changes.append(row)

    # Updates
    if not base_df.empty:
        for _, row in base_df.sample(min(10, len(base_df))).iterrows():
            modified = row.to_dict()
            modified['change_type'] = 'U'
            modified['last_modified'] = now
            all_changes.append(modified)

        # Deletes
        for id_val in base_df.sample(min(5, len(base_df)))[id_col].tolist():
            del_row = {id_col: id_val, 'change_type': 'D', 'last_modified': now}
            all_changes.append(del_row)

    # Save combined delta
    pd.DataFrame(all_changes).to_csv(f"{OUTPUT_DIR}/{name}.csv", index=False)

# Generators with real-time/transactional enhancements
def customer_row(i):
    now = datetime.now()
    return {
        'customer_id': i,
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'email': fake.email(),
        'registration_date': fake.date_between('-5y', 'today'),
        'phone': fake.phone_number(),
        'last_login': fake.date_time_between('-30d', 'now'),
        'is_active': fake.boolean()
    }

def address_row(i):
    return {
        'customer_id': i,
        'street': fake.street_address(),
        'city': fake.city(),
        'state': fake.state(),
        'zipcode': fake.zipcode(),
        'country': fake.country()
    }

def supplier_row(i):
    return {
        'supplier_id': i,
        'supplier_name': fake.company(),
        'country': fake.country(),
        'contact_number': fake.phone_number()
    }

def category_row(i):
    return {
        'category_id': i,
        'category_name': fake.word(),
        'description': fake.sentence()
    }

def product_row(i):
    return {
        'product_id': i,
        'product_name': fake.catch_phrase(),
        'category_id': random.randint(1, NUM_CATEGORIES),
        'price': round(random.uniform(5, 500), 2),
        'supplier_id': random.randint(1, NUM_SUPPLIERS),
        'is_active': fake.boolean(),
        'added_on': fake.date_time_between('-2y', 'now')
    }

def inventory_row(i):
    return {
        'product_id': i,
        'stock_quantity': random.randint(0, 1000),
        'last_updated': datetime.now()
    }

def order_row(i):
    ts = fake.date_time_between('-1y', 'now')
    return {
        'order_id': i,
        'customer_id': random.randint(1, NUM_CUSTOMERS),
        'order_date': ts,
        'order_status': random.choice(['shipped', 'cancelled', 'pending', 'returned']),
        'total_amount': round(random.uniform(10.0, 1500.0), 2),
        'payment_status': random.choice(['paid', 'unpaid', 'refunded'])
    }

def order_item_row(i):
    return {
        'item_id': i,
        'order_id': random.randint(1, NUM_ORDERS),
        'product_id': random.randint(1, NUM_PRODUCTS),
        'quantity': random.randint(1, 5),
        'item_price': round(random.uniform(5.0, 300.0), 2),
        'discount_applied': round(random.uniform(0, 30), 2)
    }

def payment_row(i):
    return {
        'payment_id': i,
        'order_id': random.randint(1, NUM_ORDERS),
        'payment_method': random.choice(['card', 'cash', 'upi', 'crypto']),
        'payment_amount': round(random.uniform(10.0, 2000.0), 2),
        'payment_date': fake.date_time_between('-1y', 'now'),
        'is_successful': fake.boolean()
    }

def shipping_row(i):
    ship_date = fake.date_time_between('-1y', 'now')
    return {
        'shipment_id': i,
        'order_id': random.randint(1, NUM_ORDERS),
        'ship_date': ship_date,
        'delivery_date': maybe_null(fake.date_time_between(start_date=ship_date, end_date='now')),
        'status': random.choice(['in_transit', 'delivered', 'delayed', 'lost'])
    }

def return_row(i):
    return {
        'return_id': i,
        'item_id': random.randint(1, NUM_ORDER_ITEMS),
        'reason': random.choice(['damaged', 'not_as_described', 'late_delivery', 'changed_mind']),
        'return_date': fake.date_time_between('-1y', 'now')
    }

def review_row(i):
    return {
        'review_id': i,
        'customer_id': random.randint(1, NUM_CUSTOMERS),
        'product_id': random.randint(1, NUM_PRODUCTS),
        'rating': random.randint(1, 5),
        'review_text': fake.text(100),
        'review_date': fake.date_time_between('-2y', 'now'),
        'verified_purchase': fake.boolean()
    }

def discount_row(i):
    return {
        'discount_id': i,
        'code': f"CODE{i}",
        'percent': round(random.uniform(5.0, 50.0), 2),
        'valid_until': fake.date_time_between('-30d', '+60d'),
        'is_active': fake.boolean()
    }

def browsing_row(i):
    return {
        'session_id': i,
        'customer_id': random.randint(1, NUM_CUSTOMERS),
        'product_id': random.randint(1, NUM_PRODUCTS),
        'timestamp': fake.date_time_between('-90d', 'now'),
        'device': random.choice(['mobile', 'desktop', 'tablet'])
    }

def support_row(i):
    return {
        'ticket_id': i,
        'customer_id': random.randint(1, NUM_CUSTOMERS),
        'created_at': fake.date_time_between('-1y', 'now'),
        'status': random.choice(['open', 'closed', 'in_progress']),
        'issue_description': fake.sentence(),
        'priority': random.choice(['low', 'medium', 'high'])
    }

def geo_row(i):
    return {
        'region_id': i,
        'country': fake.country(),
        'city': fake.city(),
        'latitude': fake.latitude(),
        'longitude': fake.longitude()
    }

# Execute generation
entities = [
    ("customers", "customer_id", customer_row),
    ("addresses", "customer_id", address_row),
    ("suppliers", "supplier_id", supplier_row),
    ("categories", "category_id", category_row),
    ("products", "product_id", product_row),
    ("inventory", "product_id", inventory_row),
    ("orders", "order_id", order_row),
    ("order_items", "item_id", order_item_row),
    ("payments", "payment_id", payment_row),
    ("shipping", "shipment_id", shipping_row),
    ("returns", "return_id", return_row),
    ("reviews", "review_id", review_row),
    ("discounts", "discount_id", discount_row),
    ("browsing_history", "session_id", browsing_row),
    ("support_tickets", "ticket_id", support_row),
    ("geographical_data", "region_id", geo_row),
]

for name, id_col, gen_func in entities:
    full_path = f"{OUTPUT_DIR}/{name}.csv"
    if MODE == "full" or not os.path.exists(full_path):
        print(f"Generating full data for {name}")
        default_size = globals().get(f"NUM_{name.upper()}", 100)
        df = pd.DataFrame([gen_func(i) for i in range(1, default_size + 1)])
        df.to_csv(full_path, index=False)
    else:
        print(f"Generating incremental data for {name}")
        existing = pd.read_csv(full_path)
        incremental_template(name, existing, gen_func, id_col)

print("Data generation complete.")


Generating full data for customers
Generating full data for addresses
Generating full data for suppliers
Generating full data for categories
Generating full data for products
Generating full data for inventory
Generating full data for orders
Generating full data for order_items
Generating full data for payments
Generating full data for shipping
Generating full data for returns
Generating full data for reviews
Generating full data for discounts
Generating full data for browsing_history
Generating full data for support_tickets
Generating full data for geographical_data
Data generation complete.
